In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [2]:
df = pd.read_csv("car_repair_dataset_with_part_names (3).csv")

df

,Record_ID,Car_Model,Model_Year,Mileage_KM,Fault_Category,Fault_Type,Fault_Code,Fault_Name,Severity,Parts_Availability,Garage_Type,Mechanic_Expertise,Day_of_Week,Time_of_Day,Actual_Repair_Hours,Repair_Cost_LKR,PartsRequired_Names
0,1,Toyota Corolla,2011,106305,Transmission,Clutch Wear,TRA-196,Clutch Wear Issue,Low,In Stock,Local,Senior,Weekend,Afternoon,1.05,5000,"Gearbox Mount, Clutch Plate"
1,2,Nissan Sunny,2017,119955,Electrical,Alternator Failure,ELE-439,Alternator Failure Issue,High,In Stock,Authorized,Senior,Weekend,Morning,6.02,27200,"Battery, Starter Motor"
2,3,Suzuki WagonR,2017,153406,Electrical,Wiring Fault,ELE-831,Wiring Fault Issue,Low,In Stock,Authorized,Expert,Weekday,Evening,2.58,14400,"Alternator, Battery"
3,4,Honda Fit,2020,166565,Electrical,Alternator Failure,ELE-439,Alternator Failure Issue,High,In Stock,Local,Expert,Weekend,Afternoon,6.18,34700,"Alternator, Battery"
4,5,Toyota Prius,2011,140270,Transmission,Clutch Wear,TRA-196,Clutch Wear Issue,High,In Stock,Authorized,Junior,Weekend,Evening,7.20,29100,"Gearbox Mount, Transmission Fluid Filter"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,Honda Fit,2022,77449,Electrical,Alternator Failure,ELE-439,Alternator Failure Issue,High,In Stock,Local,Senior,Weekday,Afternoon,5.81,21900,"Wiring Harness, Alternator"
996,997,Suzuki WagonR,2010,84533,Brake,Brake Pad Wear,BRA-207,Brake Pad Wear Issue,Medium,Order Required,Local,Junior,Weekday,Morning,4.59,21800,"Brake Caliper, Brake Fluid Hose"
997,998,Honda Civic,2022,28618,Transmission,Gear Slip,TRA-571,Gear Slip Issue,High,In Stock,Local,Senior,Weekend,Afternoon,7.01,36500,"Clutch Plate, Gearbox Mount"
998,999,Toyota Prius,2019,128086,Engine,Overheating,ENG-450,Overheating Issue,Medium,In Stock,Local,Expert,Weekday,Afternoon,4.28,15300,"Timing Belt, Engine Oil Filter"


In [3]:
features = [
    'Car_Model', 
    'Model_Year', 
    'Mileage_KM', 
    'Fault_Type', 
    'Severity',             # High/Low severity drastically changes cost
    'Actual_Repair_Hours',  # The #1 predictor of cost
    'Garage_Type',
    'Parts_Availability'
    ]

In [4]:
X = df[features]
y = df['Repair_Cost_LKR']

In [5]:
X_encoded = pd.get_dummies(X, drop_first=True)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

In [7]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [8]:
score = model.score(X_test, y_test) 

In [9]:
print(f"New R^2 Score: {score:.4f}") 

New R^2 Score: 0.8557


In [10]:
import joblib
import numpy as np
from sklearn.metrics import mean_squared_error
y_pred = model.predict(X_test)

In [11]:
rmse = np.sqrt(mean_squared_error(y_test,y_pred))
print("RMSE:", rmse)

RMSE: 3712.8462215664144


In [12]:
joblib.dump({
    "model": model,
    "cat_encoder": X_encoded,
     "rmse": rmse
    }, "part_price_model_predictcost3.pkl")


['part_price_model_predictcost3.pkl']

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    })

In [17]:
results_df = pd.DataFrame(results)
results_df.sort_values("RMSE")

,Model,MAE,RMSE,R2
0,Linear Regression,2637.100216,3367.249857,0.881325
2,Gradient Boosting,2717.181152,3516.859672,0.870545
1,Random Forest,2847.160000,3711.997217,0.855780
